# 🔑 Resend Beta Key Webhook — End-to-End Workflow

This notebook demonstrates the **Resend webhook pipeline** that fires when a beta license key email is sent with subject:

> **"🔑 Your Free Beta License Key — Hedge Edge"**

### What the handler does:
1. **Captures** email + first name from the Resend `email.sent` webhook payload
2. **Adds/updates** the user in Notion's `beta_waitlist` database
3. **Checks** if the user exists in `leads_crm` — adds them if not
4. **Posts** a Discord alert to `#log-alerts` channel

### Architecture:
```
Resend (email.sent event) → Webhook endpoint → Handler script
    ├── Notion: beta_waitlist (add/update)
    ├── Notion: leads_crm (check/add)
    └── Discord: #log-alerts (embed alert)
```

In [ ]:
# ── Section 1: Install & Import Required Libraries ──
import sys, os, json, re
from datetime import datetime, timezone

# Set workspace root for imports
WS_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
if os.path.isdir(os.path.join(WS_ROOT, "shared")):
    sys.path.insert(0, WS_ROOT)
else:
    # Fallback: try common path
    WS_ROOT = r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge"
    sys.path.insert(0, WS_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(WS_ROOT, ".env"), override=True)

from shared.resend_client import send_email, list_webhooks, create_webhook
from shared.notion_client import add_row, query_db, update_row, DATABASES
from shared.discord_client import send_embed

print(f"✅ All imports loaded — workspace: {WS_ROOT}")
print(f"   Notion DBs available: {len(DATABASES)}")
print(f"   beta_waitlist ID: {DATABASES['beta_waitlist'][:12]}...")
print(f"   leads_crm ID:     {DATABASES['leads_crm'][:12]}...")

## Section 2 — Configure API Keys & Constants

Environment variables are loaded from `.env` at workspace root. Key constants:
- **Target subject**: `"🔑 Your Free Beta License Key — Hedge Edge"`
- **Discord channel**: `#log-alerts` (ID: `1477095672800083989`)
- **Test email**: `hedgedge@hedgedge.info`

In [ ]:
# ── Section 2: Constants & Config ──
TARGET_SUBJECT = "🔑 Your Free Beta License Key — Hedge Edge"
DISCORD_LOG_ALERTS_CHANNEL = "1477095672800083989"
TEST_EMAIL = "hedgedge@hedgedge.info"
TEST_NAME = "TestUser"
TODAY = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Verify API keys are loaded (without printing them)
keys = {
    "RESEND_API_KEY": bool(os.getenv("RESEND_API_KEY")),
    "NOTION_API_KEY": bool(os.getenv("NOTION_API_KEY") or os.getenv("NOTION_TOKEN")),
    "DISCORD_BOT_TOKEN": bool(os.getenv("DISCORD_BOT_TOKEN")),
}
for k, v in keys.items():
    print(f"  {'✅' if v else '❌'} {k}: {'loaded' if v else 'MISSING'}")

print(f"\n  Target subject: {TARGET_SUBJECT}")
print(f"  Test email:     {TEST_EMAIL}")

## Section 3 — Set Up Notion Client & Verify Connectivity

Verify we can reach both Notion databases by querying them.

In [ ]:
# ── Section 3: Verify Notion DB connectivity ──
print("Querying beta_waitlist...")
waitlist_rows = query_db("beta_waitlist")
print(f"  ✅ beta_waitlist: {len(waitlist_rows)} rows")

print("Querying leads_crm...")
leads_rows = query_db("leads_crm")
print(f"  ✅ leads_crm: {len(leads_rows)} rows")

## Section 4 — Capture User Email & First Name

Define the helper functions that extract user details from Resend's webhook payload. The `to` field comes as `["FirstName <email@example.com>"]` or plain `["email@example.com"]`.

In [ ]:
# ── Section 4: User detail extraction from webhook payload ──

def extract_email(to_field) -> str:
    """Extract plain email from Resend 'to' field."""
    addr = to_field[0] if isinstance(to_field, list) and to_field else str(to_field)
    if "<" in addr and ">" in addr:
        return addr.split("<")[1].split(">")[0].strip()
    return addr.strip()

def extract_first_name(to_field) -> str:
    """Extract first name from Resend 'to' field."""
    addr = to_field[0] if isinstance(to_field, list) and to_field else str(to_field)
    if "<" in addr:
        name_part = addr.split("<")[0].strip()
        if name_part:
            return name_part.split()[0]
    email = addr.strip().strip("<>")
    return email.split("@")[0].capitalize()

def validate_email(email: str) -> bool:
    """Basic email format validation."""
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email))

# Test with mock data
test_to = [f"{TEST_NAME} <{TEST_EMAIL}>"]
email = extract_email(test_to)
first_name = extract_first_name(test_to)
print(f"  Extracted email:      {email}")
print(f"  Extracted first name: {first_name}")
print(f"  Email valid:          {validate_email(email)}")

## Section 5 — Check if User Exists in Leads Database

Query `leads_crm` by email to determine if the user is already a known lead.

In [ ]:
# ── Section 5: Check leads_crm for existing user ──

def find_in_leads(email: str) -> tuple[bool, dict | None]:
    """Check if user exists in leads_crm. Returns (exists, row_or_None)."""
    rows = query_db("leads_crm")
    email_lower = email.lower().strip()
    for row in rows:
        if (row.get("Email") or "").lower().strip() == email_lower:
            return True, row
    return False, None

exists, lead_row = find_in_leads(TEST_EMAIL)
print(f"  User {TEST_EMAIL} in leads_crm: {exists}")
if lead_row:
    print(f"  Lead record: {lead_row.get('Subject', '?')} | Source: {lead_row.get('Source', '?')}")

## Section 6 — Add User to Leads Database if Not Present

If the user was **not** found in `leads_crm`, create a new lead with source `"Beta Waitlist"`.

In [ ]:
# ── Section 6: Add to leads_crm if not present ──

def ensure_in_leads(email: str, first_name: str) -> dict:
    """Add user to leads_crm if they don't already exist."""
    exists, row = find_in_leads(email)
    if exists:
        print(f"  ℹ️  {email} already in leads_crm — skipping")
        return {"action": "already_exists", "page_id": row["_id"]}

    print(f"  ➕  Adding {email} to leads_crm")
    # Schema: Subject (title), First Name, Email, Source
    result = add_row("leads_crm", {
        "Subject": first_name or email.split("@")[0],
        "First Name": first_name,
        "Email": email,
        "Source": "Beta Waitlist",
    })
    return {"action": "created", "page_id": result.get("id")}

leads_result = ensure_in_leads(TEST_EMAIL, TEST_NAME)
print(f"  Result: {leads_result}")

## Section 7 — Add User to Beta Waitlist Database

Add/update the user in `beta_waitlist`. If they already exist, just update `Beta Key Sent` and `Key Sent Date`. If new, create a full entry.

In [ ]:
# ── Section 7: Add/update beta_waitlist ──

def find_in_beta_waitlist(email: str) -> tuple[bool, dict | None]:
    """Check if user exists in beta_waitlist."""
    rows = query_db("beta_waitlist")
    email_lower = email.lower().strip()
    for row in rows:
        if (row.get("Email") or "").lower().strip() == email_lower:
            return True, row
    return False, None

def add_to_beta_waitlist(email: str, first_name: str) -> dict:
    """Add new entry or update existing in beta_waitlist."""
    exists, row = find_in_beta_waitlist(email)
    if exists:
        print(f"  ℹ️  {email} already in beta_waitlist — updating Beta Key Sent")
        update_row(row["_id"], "beta_waitlist", {
            "Beta Key Sent": True,
            "Key Sent Date": TODAY,
        })
        return {"action": "updated", "page_id": row["_id"]}

    print(f"  ➕  Adding {email} to beta_waitlist")
    result = add_row("beta_waitlist", {
        "Name": first_name or email.split("@")[0],
        "First Name": first_name,
        "Email": email,
        "Source": "Landing Page",
        "Priority": "P2",
        "Beta Key Sent": True,
        "Key Sent Date": TODAY,
        "Lifecycle Owner": "Marketing",
    })
    return {"action": "created", "page_id": result.get("id")}

waitlist_result = add_to_beta_waitlist(TEST_EMAIL, TEST_NAME)
print(f"  Result: {waitlist_result}")

## Section 8 — Send Beta License Key Email via Resend

Send the beta key email using the Resend API. The email subject must match exactly:
> `"🔑 Your Free Beta License Key — Hedge Edge"`

This is what triggers the webhook on the Resend side.

In [ ]:
# ── Section 8: Send beta key email via Resend ──
import secrets

def generate_beta_key() -> str:
    """Generate a unique beta license key: XXXXX-XXXXX-XXXXX-XXXXX-XXXXX"""
    segments = [secrets.token_hex(3).upper()[:5] for _ in range(5)]
    return "-".join(segments)

beta_key = generate_beta_key()
print(f"  Generated beta key: {beta_key}")

# Send the email
result = send_email(
    to=TEST_EMAIL,
    subject=TARGET_SUBJECT,
    html=f"""
    <div style="font-family:sans-serif;max-width:600px;margin:0 auto;">
      <h2>🔑 Your Beta License Key</h2>
      <p>Hi {TEST_NAME},</p>
      <p>Your unique beta key: <strong>{beta_key}</strong></p>
      <p>Activate within 48 hours by pasting it into the Hedge Edge desktop app.</p>
      <p>— Hedge Edge Team</p>
    </div>
    """,
    tags=[{"name": "category", "value": "beta_key"}],
)
print(f"  ✅ Email sent! ID: {result.get('id', 'unknown')}")

## Section 9 — Send Discord Alert to #log-alerts

Post a rich embed to the `#log-alerts` Discord channel with the user's details.

In [ ]:
# ── Section 9: Discord alert to #log-alerts ──

def send_discord_alert(email: str, first_name: str, waitlist_action: str, leads_action: str):
    """Post a rich embed alert to #log-alerts."""
    fields = [
        {"name": "Email", "value": email, "inline": True},
        {"name": "First Name", "value": first_name or "(unknown)", "inline": True},
        {"name": "Waitlist", "value": waitlist_action, "inline": True},
        {"name": "Leads CRM", "value": leads_action, "inline": True},
        {"name": "Timestamp", "value": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"), "inline": False},
    ]
    send_embed(
        channel_id=DISCORD_LOG_ALERTS_CHANNEL,
        title="🔑 Beta Key Email Sent",
        description=f"Resend confirmed beta key email delivered to **{email}**",
        color=0x22C55E,
        fields=fields,
    )
    print(f"  ✅ Discord alert sent to #log-alerts")

send_discord_alert(TEST_EMAIL, TEST_NAME, waitlist_result["action"], leads_result["action"])

## Section 10 — End-to-End Test with Full Handler

Run the complete `handle_webhook_event()` function from the production script with a mock payload. This exercises the full pipeline in one call. Running twice verifies duplicate detection.

In [ ]:
# ── Section 10: End-to-End test using the production handler ──
from Business.GROWTH.executions.Marketing.resend_beta_key_webhook import handle_webhook_event

# Build mock Resend webhook payload
mock_event = {
    "type": "email.sent",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "data": {
        "email_id": "test-notebook-e2e",
        "from": "Hedge Edge <hello@hedgedge.info>",
        "to": [f"{TEST_NAME} <{TEST_EMAIL}>"],
        "subject": TARGET_SUBJECT,
        "tags": {"category": "beta_key"},
    },
}

print("━━━ Run 1: Should add/update ━━━")
result1 = handle_webhook_event(mock_event)
print(f"\nResult: {json.dumps(result1, indent=2)}")

print("\n━━━ Run 2: Should detect duplicates ━━━")
result2 = handle_webhook_event(mock_event)
print(f"\nResult: {json.dumps(result2, indent=2)}")

# Verify
print("\n━━━ Verification ━━━")
_, wl = find_in_beta_waitlist(TEST_EMAIL)
_, ld = find_in_leads(TEST_EMAIL)
print(f"  beta_waitlist: {'✅ found' if wl else '❌ missing'}")
print(f"  leads_crm:     {'✅ found' if ld else '❌ missing'}")
if wl:
    print(f"    Beta Key Sent: {wl.get('Beta Key Sent')}")
    print(f"    Key Sent Date: {wl.get('Key Sent Date')}")

## Section 11 — Check Existing Resend Webhooks

List all currently registered Resend webhooks to see what's already set up.

In [ ]:
# ── Section 11: List existing Resend webhooks ──
webhooks = list_webhooks()
print(f"Registered webhooks: {len(webhooks)}\n")
for wh in webhooks:
    print(f"  ID:       {wh['id']}")
    print(f"  Endpoint: {wh['endpoint']}")
    print(f"  Events:   {wh['events']}")
    print(f"  Status:   {wh.get('status', '?')}")
    print()

## Section 12 — Register Production Webhook on Resend

**⚠️ Run this cell only once** — after you deploy the webhook handler endpoint.

Replace `YOUR_ENDPOINT_URL` with the production URL (e.g. a Vercel API route or Supabase edge function). The signing secret returned should be stored securely for signature verification.

In [ ]:
# ── Section 12: Register the production webhook (run once) ──
# ⚠️  UNCOMMENT the lines below when you have a production endpoint ready.
#
# ENDPOINT_URL = "https://YOUR_ENDPOINT_URL/api/resend-beta-webhook"
#
# result = create_webhook(
#     endpoint=ENDPOINT_URL,
#     events=["email.sent"],
# )
# print(f"✅ Webhook registered!")
# print(f"   ID:             {result.get('id')}")
# print(f"   Signing Secret: {result.get('signing_secret')}")
# print(f"\n   ⚠️  Store the signing_secret securely — it's shown only once.")

print("ℹ️  Webhook registration is commented out. Uncomment when endpoint is deployed.")

## Summary & Next Steps

### ✅ What's been built and tested:
- **Webhook handler** (`resend_beta_key_webhook.py`) — processes `email.sent` events for the beta key subject
- **Notion integration** — adds to `beta_waitlist` + creates lead in `leads_crm` if absent
- **Discord alerts** — posts rich embeds to `#log-alerts`
- **Duplicate detection** — re-runs safely without creating duplicate records

### 🚀 Next steps to go live:
1. **Deploy the handler** as a Vercel API route (`/api/resend-beta-webhook`) or Supabase edge function
2. **Register the webhook** via Section 12 above with the production endpoint URL
3. **Store the `signing_secret`** in `.env` for payload signature verification
4. **Monitor** `#log-alerts` on Discord for real beta key sends